<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Solution: Cross-encoder re-ranking

**[Cross-encoder re-ranking](https://learning.nextia-ai.com/courses/search/m06/cross-encoder-re-ranking/)** · Search and Ranking: Find the Best Match · Module 6, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** re-rank candidates with recorded cross-encoder scores, measure the effect by language, and work out how many candidates fit a latency budget.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 9 MB). No account, no key and no model download. |
| **You should know** | Two-stage ranking, from [Two-stage ranking](https://learning.nextia-ai.com/courses/search/m06/two-stage-ranking/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m06/cross-encoder-re-ranking/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M06-L02-cross-encoder-re-ranking/cross-encoder-re-ranking-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 10 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/recorded/rerank.csv.gz": "38971117046aefd883c159ed9a7b196a0ca88f54757a8dc35df956d39222da88",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Read the query and the product together

A **cross-encoder** is a language model that reads the query and one product's text together, as one input, and gives one score: how well the product answers the query. It sees every word of both at once, so it understands phrases such as "beds that have leds". The price: one model run for every pair.

The course team ran the cross-encoder (IBM's granite-embedding-reranker-english-r2) once on the top 100 candidates of hybrid search for every validation and test query, and recorded the scores. So this notebook needs no model download. Run the next cell.

In [ ]:
q = queries()
j = judgments()
rr = pd.read_csv("data/recorded/rerank.csv.gz")      # query_id, product_id, hybrid_rank, score
print(rr["query_id"].nunique(), "queries,", len(rr), "query-product pairs with a recorded score")

## 2. A worked example

> **Predict.** Hybrid search puts LED light bulbs first for "beds that have leds". Will the cross-encoder put beds first?

In [ ]:
# One test query: "beds that have leds" (W-014). Hybrid order first, then the cross-encoder's order.
g = rr[rr["query_id"] == "W-014"].sort_values("hybrid_rank")
print("Hybrid search, top 5:")
show(g["product_id"], j["W-014"], n=5)
print()
print("Cross-encoder on the top 50, top 5:")
top50 = g.head(50).sort_values(["score", "product_id"], ascending=[False, True])
show(top50["product_id"], j["W-014"], n=5)
print()
print(g.head(3)[["hybrid_rank", "product_id", "score"]].to_string(index=False))
print(top50.head(3)[["hybrid_rank", "product_id", "score"]].to_string(index=False))

The scores are close together (about 0.75 to 0.88), but their **order** is what matters. The light bulbs score about 0.76; the bed with LEDs scores higher. The cross-encoder read "beds" as the thing and "leds" as a detail.

## 3. Does it help every language?

The next cell re-ranks the top 20 candidates of every validation query (20 was the best depth in [Two-stage ranking](https://learning.nextia-ai.com/courses/search/m06/two-stage-ranking/)) and compares NDCG@10 with hybrid search, query by query. Then it averages the difference by language.

In [ ]:
def rerank(query_ids, n):
    """Stage one: the hybrid order. Stage two: the cross-encoder re-orders the first n candidates."""
    out = {}
    for qid, g in rr[rr["query_id"].isin(query_ids)].groupby("query_id"):
        g = g.sort_values("hybrid_rank")
        head = g.head(n).sort_values(["score", "product_id"], ascending=[False, True])
        out[qid] = list(head["product_id"]) + list(g["product_id"].iloc[n:])
    return out

# Validation queries without the product codes (the code rule puts their product first before any ranking).
val = q[(q["split"] == "validation") & (q["source"] != "code")]
before = cp.per_query(rerank(set(val["query_id"]), 0), j)
after = cp.per_query(rerank(set(val["query_id"]), 20), j)
diff = (after["ndcg@10"] - before["ndcg@10"]).rename("difference")
by_lang = diff.to_frame().join(val.set_index("query_id")["language"]).groupby("language")["difference"].agg(["count", "mean"])
by_lang

## 4. Your turn: the French queries

In [ ]:
fr = val.loc[val["language"] == "fr", "query_id"]
fr_diff = float(diff.loc[diff.index.isin(fr)].mean())
print(round(fr_diff, 3))

In [ ]:
expect('fr_diff', fr_diff, -0.01900608812613268, tolerance=0.0005)

> **Check.** On the 81 English validation queries, re-ranking changes NDCG@10 by 0.016 on average; on the 30 French queries by -0.019. The model's card says it was trained for English. In the lesson, it reads "chaise" in a French query as the English "chaise lounge", not as the French word for "chair".

In [ ]:
# The French queries that re-ranking hurt most, with their English original (base_query_id).
worst = diff.loc[diff.index.isin(fr)].sort_values().head(5)
print(worst.to_frame().join(q.set_index("query_id")[["query", "base_query_id"]]).to_string())

## 5. Your turn: what fits in 200 ms?

Amira's budget for a whole search is 200 ms. Hybrid search takes about 10 ms. The cross-encoder scored 75.7 pairs a second on 4 CPU threads.

In [ ]:
pairs_per_second = 75.7      # measured: the cross-encoder on 4 threads, batch 32, 256 tokens (Apple M4)
stage_one_ms = 10            # measured: hybrid search, p50 per query
budget_ms = 200
n_fit = int((budget_ms - stage_one_ms) / 1000 * pairs_per_second)
print(n_fit)

In [ ]:
expect('n_fit', n_fit, 14)

> **Check.** About 14 pairs fit. The live measurements agree: 10 candidates took 155 ms per search, and 20 took 295 ms. Re-ranking 50 candidates takes about 0.7 seconds, more than three times the budget. In the project: `search eval --run rerank-ce --set rerank=10`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Learning to rank](https://learning.nextia-ai.com/courses/search/m06/learning-to-rank/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m06/cross-encoder-re-ranking/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.